# HotpotQA Multi-Trial: Reflexion vs Structured Correction


In [ ]:
import sys, os
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')
key = os.environ.get('OPENAI_API_KEY','').strip().strip('"').strip("'")
os.environ['OPENAI_API_KEY'] = key
for _k in ['OPENAI_API_BASE','OPENAI_API_TYPE','OPENAI_API_VERSION',
           'AZURE_OPENAI_ENDPOINT','AZURE_OPENAI_API_KEY']:
    os.environ.pop(_k, None)
assert key, 'Set OPENAI_API_KEY in .env'
print('API key loaded, length =', len(key))

In [ ]:
import importlib
import agents as _a; importlib.reload(_a)
import prompts as _p; importlib.reload(_p)

import joblib
from util import summarize_react_trial, log_react_trial, save_agents
from agents import ReactReflectAgent, ReactAgent, ReflexionStrategy
from agents import is_useful_reflection, is_duplicate_reflection, AnyOpenAILLM

print('agents from:', _a.__file__)
print('Strategies :', [s.value for s in ReflexionStrategy])

required = ['REFLEXION_FILTERED', 'DC_3LINE_FILTERED', 'SAFE_DC_3LINE']
for r in required:
    assert hasattr(ReflexionStrategy, r), f'MISSING: {r} - update agents.py'
print('All new strategies present.')

In [ ]:
N_QUESTIONS = 100   
N_TRIALS    = 5      
TEMPERATURE = 0.7   
root        = os.path.join('..', 'root')
os.makedirs(root, exist_ok=True)

STRATEGIES = [
    ReflexionStrategy.NONE,
    ReflexionStrategy.REFLEXION,
    ReflexionStrategy.DIALOGUE_CORRECTION_3LINE,
    ReflexionStrategy.REFLEXION_FILTERED,
    ReflexionStrategy.DC_3LINE_FILTERED,
    ReflexionStrategy.SAFE_DC_3LINE,
]
LABELS = {
    ReflexionStrategy.NONE:                      'NONE',
    ReflexionStrategy.REFLEXION:                 'Reflexion (Shinn et al.)',
    ReflexionStrategy.DIALOGUE_CORRECTION_3LINE: 'DC-3line',
    ReflexionStrategy.REFLEXION_FILTERED:        'Reflexion-Filtered (pivot 1)',
    ReflexionStrategy.DC_3LINE_FILTERED:         'DC-3line-Filtered (pivot 1)',
    ReflexionStrategy.SAFE_DC_3LINE:             'Safe-DC-3line (pivot 2)',
}
COLORS = {
    ReflexionStrategy.NONE:                      '#d62728',
    ReflexionStrategy.REFLEXION:                 '#ff7f0e',
    ReflexionStrategy.DIALOGUE_CORRECTION_3LINE: '#2ca02c',
    ReflexionStrategy.REFLEXION_FILTERED:        '#1f77b4',
    ReflexionStrategy.DC_3LINE_FILTERED:         '#9467bd',
    ReflexionStrategy.SAFE_DC_3LINE:             '#17becf',
}
# Result buckets — shared across all per-strategy cells
all_results  = {}
all_logs     = {}
filter_stats = {}
print(f'Questions={N_QUESTIONS}  Trials={N_TRIALS}  Temp={TEMPERATURE}')
print('Strategies:', [s.value for s in STRATEGIES])

In [ ]:
hotpot = joblib.load('../data/hotpot-qa-distractor-sample.joblib').reset_index(drop=True)
hotpot = hotpot.head(N_QUESTIONS)
print(f'Loaded {len(hotpot)} questions')

def make_react_agent(q, a):
    return ReactAgent(
        q, a,
        react_llm=AnyOpenAILLM(
            model_name='gpt-3.5-turbo',
            temperature=TEMPERATURE,
            max_tokens=100,
            model_kwargs={'stop': '\n'},
            openai_api_key=os.environ['OPENAI_API_KEY'],
        ),
    )

def make_reflect_agent(q, a):
    return ReactReflectAgent(
        q, a,
        react_llm=AnyOpenAILLM(
            model_name='gpt-3.5-turbo',
            temperature=TEMPERATURE,
            max_tokens=100,
            model_kwargs={'stop': '\n'},
            openai_api_key=os.environ['OPENAI_API_KEY'],
        ),
        reflect_llm=AnyOpenAILLM(
            model_name='gpt-3.5-turbo',
            temperature=TEMPERATURE,
            max_tokens=250,
            openai_api_key=os.environ['OPENAI_API_KEY'],
        ),
    )

def run_strategy(strategy):
    print(f'\n{"="*62}')
    print(f'  {LABELS[strategy]}')
    print(f'{"="*62}')
    
    if strategy == ReflexionStrategy.NONE:
        agents = [make_react_agent(r['question'], r['answer'])
                  for _, r in hotpot.iterrows()]
    else:
        agents = [make_reflect_agent(r['question'], r['answer'])
                  for _, r in hotpot.iterrows()]
    
    trial_accs = []
    log = ''
    stored = discarded = 0
    
    for trial_n in range(1, N_TRIALS + 1):
        n_to_run = sum(1 for a in agents if not a.is_correct())
        print(f'  Trial {trial_n}: running {n_to_run} agents...', flush=True)
        
        for agent in [a for a in agents if not a.is_correct()]:
            prev_mem_len = len(getattr(agent, 'correction_memory', [])
                               + getattr(agent, 'reflections', []))
            
            if strategy == ReflexionStrategy.NONE:
                agent.run()
            else:
                agent.run(reflect_strategy=strategy)
            
            new_mem_len = len(getattr(agent, 'correction_memory', [])
                              + getattr(agent, 'reflections', []))
            if new_mem_len > prev_mem_len:
                stored += 1
            elif strategy not in (ReflexionStrategy.NONE,
                                  ReflexionStrategy.REFLEXION,
                                  ReflexionStrategy.DIALOGUE_CORRECTION_3LINE):
                discarded += 1
        
        correct, incorrect, halted = summarize_react_trial(agents)
        acc = len(correct) / len(agents)
        trial_accs.append(acc)
        log += log_react_trial(agents, trial_n)
        print(f'  Trial {trial_n}: correct={len(correct):3d} '
              f'incorrect={len(incorrect):3d} halted={len(halted):3d}  acc={acc:.2f}')
    
    all_results[strategy]  = trial_accs
    all_logs[strategy]     = log
    filter_stats[strategy] = {'stored': stored, 'discarded': discarded}
    
    log_dir = os.path.join(root, 'ReAct', strategy.value)
    os.makedirs(log_dir, exist_ok=True)
    fname = f'{N_QUESTIONS}q_{N_TRIALS}t_temp{TEMPERATURE}.txt'
    with open(os.path.join(log_dir, fname), 'w', encoding='utf-8') as f:
        f.write(log)
    print(f'  Saved to {log_dir}')

print('Helper function defined.')

---
## Run each strategy independently

Run these one at a time. You can pause between cells and come back.

### Strategy 1: NONE (baseline)

In [ ]:
run_strategy(ReflexionStrategy.NONE)

### Strategy 2: REFLEXION (Shinn et al. freeform)

In [ ]:
run_strategy(ReflexionStrategy.REFLEXION)

### Strategy 3: DC-3line (structured 3-line correction)

In [ ]:
run_strategy(ReflexionStrategy.DIALOGUE_CORRECTION_3LINE)

### Strategy 4: Reflexion-Filtered (pivot 1 on freeform)

In [ ]:
run_strategy(ReflexionStrategy.REFLEXION_FILTERED)

### Strategy 5: DC-3line-Filtered (pivot 1 on structured)

In [ ]:
run_strategy(ReflexionStrategy.DC_3LINE_FILTERED)

### Strategy 6: Safe-DC-3line (pivot 2: filter + revision gate)

In [ ]:
run_strategy(ReflexionStrategy.SAFE_DC_3LINE)